In [1]:
import os
import subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import re

In [2]:
num_workers = 4  
fastq_dir = Path("./fastq")  
symlink_dir = Path("./symlinks") # Temporary clean-named links
#results_dir = Path("results")
snp_only_dir = Path("./snp_only")
logs_dir = Path("./logs")
tmp_dir = Path("./tmp")
vcf_dir = Path("./vcf")

In [3]:
#results_dir.mkdir(exist_ok=True)
fastq_dir.mkdir(exist_ok=True)
symlink_dir.mkdir(exist_ok=True)
snp_only_dir.mkdir(exist_ok=True)
logs_dir.mkdir(exist_ok=True)
tmp_dir.mkdir(exist_ok=True)
vcf_dir.mkdir(exist_ok=True)

In [4]:
# Group FASTQs by ENA run ID
samples = {}

for fq in fastq_dir.glob("*.fastq.gz"):
    name = fq.name

    if fq.name.endswith(".aria2"):
        #print(f"⚠️ Skipping {fq.name}: still downloading (.aria2).")
        continue

    # Match ERR / SRR with regex
    match = re.search(r'([SE]RR\d+)', name)
    if not match:
        print(f"⚠️ Skipping {name}: no ENA run ID found.")
        continue

    clean_base = match.group(1)

    # Figure out _1 or _2
    if "_1" in name:
        suffix = "_1"
    elif "_2" in name:
        suffix = "_2"
    else:
        suffix = ""
    # else:
    #     print(f"⚠️ Skipping {name}: no read pair info.")
    #     continue

    # Make symlink with correct name
    symlink_name = f"{clean_base}{suffix}.fastq.gz"
    symlink_path = symlink_dir / symlink_name

    if not symlink_path.exists():
        symlink_path.symlink_to(fq.resolve())

    samples.setdefault(clean_base, []).append(symlink_path)

print(f"✅ Found {len(samples)} samples to process.")

✅ Found 406 samples to process.


In [5]:
# Function to process one sample
def process_sample(sample, files):
    files = sorted(files)
    output_prefix = sample

    if len(files) == 2:
        fq1, fq2 = files
        tb_cmd = [
            "tb-profiler",
            "profile",
            "-1", str(fq1),
            "-2", str(fq2),
            "-p", str(output_prefix),
            "--txt",
            "--temp", "./tmp"
        ]
    elif len(files) == 1:
        fq1 = files[0]
        tb_cmd = [
            "tb-profiler",
            "profile",
            "-1", str(fq1),
            "-p", str(output_prefix),
            "--txt",
            "--temp", "./tmp"
        ]
    else:
        print(f"⚠️ Skipping {sample}: no valid FASTQ files.")
        return

    print(f"🔬 Running TB-Profiler for {sample}")
    with open(logs_dir / f"{sample}_tbprofiler.log", "w") as log_file:
        subprocess.run(tb_cmd, stdout=log_file, stderr=log_file, check=True)

    input_vcf = vcf_dir / f"{output_prefix}.targets.vcf.gz"
    output_vcf = snp_only_dir / f"{sample}_snps_only.vcf"

    # STEP 1 — Extract SNPs only
    bcf_cmd = [
        "bcftools", "view",
        "-v", "snps",
        input_vcf,
        "-o", str(output_vcf),
        "--output-type", "v"
    ]

    print(f"🧬 Extracting SNPs for {sample}")
    with open(logs_dir / f"{sample}_bcftools.log", "w") as log_file:
        subprocess.run(bcf_cmd, stdout=log_file, stderr=log_file, check=True)

    print(f"✅ Finished: {sample}")

    # STEP 2 — Filter SNPs with DP, QUAL, AF thresholds
    filtered_vcf = snp_only_dir / f"{sample}_snps_only_filtered.vcf"
    bcf_filter_cmd = [
        "bcftools", "filter",
        "-i", "DP>=5 && QUAL>=20 && AF>=0.75",
        str(snps_only_vcf),
        "-o", str(filtered_vcf)
    ]

    print(f"🧪 Filtering SNPs for {sample} with DP>=5, QUAL>=20, AF>=0.75")
    with open(logs_dir / f"{sample}_bcftools_filter.log", "w") as log_file:
        subprocess.run(bcf_filter_cmd, stdout=log_file, stderr=log_file, check=True)

    print(f"✅ Finished: {sample} — Final filtered SNPs: {filtered_vcf}")

In [ ]:
# Run with parallel jobs 
with ThreadPoolExecutor(max_workers=num_workers) as executor:
    futures = []
    for sample, files in samples.items():
        futures.append(executor.submit(process_sample, sample, files))

    for future in as_completed(futures):
        try:
            future.result()
        except subprocess.CalledProcessError as e:
            print(f"❌ Error running command: {e}") # meron pa ring error like ERR4810720, ERR2199934, SRR6152742 figure out why

print("🎉 All samples done.")
# 8:34 pm - terminated as no new files are being downloaded
# 9:02 pm

🔬 Running TB-Profiler for SRR6650422
⚠️ Skipping SRR6650425: no valid FASTQ files.
🔬 Running TB-Profiler for SRR6650354
⚠️ Skipping SRR6650353: no valid FASTQ files.
⚠️ Skipping SRR6650301: no valid FASTQ files.
🔬 Running TB-Profiler for SRR6650289
🔬 Running TB-Profiler for SRR6797557
⚠️ Skipping SRR6797638: no valid FASTQ files.
🔬 Running TB-Profiler for ERR5917671
❌ Error running command: Command '['tb-profiler', 'profile', '-1', 'symlinks/SRR6650289_1.fastq.gz', '-p', 'SRR6650289', '--txt', '--temp', './tmp']' returned non-zero exit status 1.
🔬 Running TB-Profiler for ERR5917706
❌ Error running command: Command '['tb-profiler', 'profile', '-1', 'symlinks/SRR6650422_1.fastq.gz', '-2', 'symlinks/SRR6650422_2.fastq.gz', '-p', 'SRR6650422', '--txt', '--temp', './tmp']' returned non-zero exit status 1.
🔬 Running TB-Profiler for ERR5917753❌ Error running command: Command '['tb-profiler', 'profile', '-1', 'symlinks/ERR5917671_1.fastq.gz', '-2', 'symlinks/ERR5917671_2.fastq.gz', '-p', 'ERR5

KeyboardInterrupt: 

⚠️ Skipping SRR2024933: no valid FASTQ files.🔬 Running TB-Profiler for SRR2024934


🔬 Running TB-Profiler for SRR2024966
🔬 Running TB-Profiler for SRR671747
🔬 Running TB-Profiler for SRR671823
🔬 Running TB-Profiler for SRR671876
🔬 Running TB-Profiler for ERR2199933
